In [3]:
import queue
import json
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import ipywidgets as widgets
from IPython.display import display
from threading import Thread

# ---- Configuration ----
SAMPLE_RATE = 16000   # Vosk small EN models typically expect 16kHz
CHANNELS = 1
BLOCKSIZE = 8000
DTYPE = "int16"

# Path should be the folder that contains model.conf, am/, conf/, etc.
model_path = r"vosk-model-small-en-us-0.15/vosk-model-small-en-us-0.15"

# ---- Load model + recognizer ----
model = Model(model_path)
rec = KaldiRecognizer(model, SAMPLE_RATE)
rec.SetWords(True)

# ---- Shared state ----
audio_q = queue.Queue()
listening = {"active": False, "stream": None, "thread": None}

def audio_callback(indata, frames, time, status):
    """sounddevice callback — push raw audio bytes to queue."""
    if status:
        # status messages are normal sometimes (buffer underrun/overrun)
        pass
    audio_q.put(bytes(indata))

def recognize_loop(output_widget, show_partials=True):
    """Read audio from queue and feed to Vosk recognizer."""
    last_partial = ""
    while listening["active"]:
        try:
            data = audio_q.get(timeout=0.1)
        except queue.Empty:
            continue

        if rec.AcceptWaveform(data):
            # Final result for the utterance that just ended
            result = json.loads(rec.Result())
            text = result.get("text", "").strip()
            if text:
                output_widget.append_stdout(text + "\n")
            last_partial = ""
        else:
            if show_partials:
                partial = json.loads(rec.PartialResult()).get("partial", "").strip()
                # Avoid spamming the same partial repeatedly
                if partial and partial != last_partial:
                    output_widget.append_stdout(partial + "\n")
                    last_partial = partial

def start_stream(_=None):
    if listening["active"]:
        return

    # Clear any old queued audio
    while not audio_q.empty():
        try:
            audio_q.get_nowait()
        except queue.Empty:
            break

    listening["active"] = True

    stream = sd.RawInputStream(
        samplerate=SAMPLE_RATE,
        blocksize=BLOCKSIZE,
        dtype=DTYPE,
        channels=CHANNELS,
        callback=audio_callback
    )
    stream.start()

    t = Thread(target=recognize_loop, args=(output, True), daemon=True)
    t.start()

    listening["stream"] = stream
    listening["thread"] = t
    output.append_stdout("[Listening started…]\n")

def stop_stream(_=None):
    if not listening["active"]:
        return

    listening["active"] = False

    stream = listening.get("stream")
    if stream:
        stream.stop()
        stream.close()

    listening["stream"] = None
    listening["thread"] = None

    # Flush any remaining recognition once at the end
    try:
        final = json.loads(rec.FinalResult()).get("text", "").strip()
        if final:
            output.append_stdout(final + "\n")
    except Exception:
        pass

    output.append_stdout("[Listening stopped]\n")

# ---- Widgets UI ----
record_button = widgets.Button(description="Start Recording", button_style="success", icon="microphone")
stop_button   = widgets.Button(description="Stop", button_style="danger", icon="stop")
output = widgets.Output(layout={'border': '1px solid black', 'height': '250px', 'overflow_y': 'auto'})

record_button.on_click(start_stream)
stop_button.on_click(stop_stream)

display(record_button, stop_button, output)


Button(button_style='success', description='Start Recording', icon='microphone', style=ButtonStyle())

Button(button_style='danger', description='Stop', icon='stop', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…